# Research 60 — HistGradientBoosting with fold-safe one-hot encoding

**Hypothesis:** sklearn HistGradientBoosting may add diversity to the CatBoost/LightGBM reference while keeping training fast. Use one fixed balanced multiclass recipe, fold-local numeric imputation and one-hot categorical encoding, 3-fold stratified CV seed 42, and only a fixed 80/20 reference blend. No parameter search or Test access.

In [1]:
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from threadpoolctl import threadpool_limits

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data/train.csv").is_file()
)
train = pd.read_csv(ROOT / "data/train.csv")
y = train.segment.to_numpy(dtype=int)


def load_probs(filename):
    frame = pd.read_csv(ROOT / "predictions" / filename)
    assert len(frame) == len(y) and np.array_equal(frame.segment.to_numpy(), y), filename
    probs = frame[[f"prob_{i}" for i in range(4)]].to_numpy(dtype=float)
    assert np.isfinite(probs).all() and (probs >= 0).all(), filename
    row_sums = probs.sum(axis=1, keepdims=True)
    assert (row_sums > 0).all(), filename
    return probs / row_sums


def make_features(frame):
    x = frame.drop(
        columns=["segment", "player_id", "random_metric_1", "random_metric_2", "random_metric_3"],
        errors="ignore",
    ).copy()
    ratios = {
        "spend_per_hour": ("total_spending_thb", "total_playtime_hours"),
        "spend_per_account_day": ("total_spending_thb", "account_age_days"),
        "playtime_per_account_day": ("total_playtime_hours", "account_age_days"),
        "monthly_spend_per_purchase": ("avg_monthly_spending", "spending_frequency"),
        "gifts_per_friend": ("gifts_sent_received", "friend_count"),
        "invites_per_friend": ("friend_invites_sent", "friend_count"),
        "ranked_per_team": ("ranked_participation_rate", "team_play_percentage"),
    }
    for name, (numerator, denominator) in ratios.items():
        x[name] = x[numerator] / (x[denominator].abs() + 1)
    x["weekly_minutes"] = x["play_frequency"] * x["avg_session_duration"]
    x["social_activity"] = x["friend_count"] * x["team_play_percentage"]
    vip = x["vip_tier"].map(lambda value: "__MISSING__" if pd.isna(value) else str(int(value)))
    region = x["region"].fillna("__MISSING__").astype(str)
    x["vip_region"] = vip + "__" + region
    x["missing_count"] = x.isna().sum(axis=1)
    return x.replace([np.inf, -np.inf], np.nan)


p_base = load_probs("segment-oof.csv")
X = make_features(train)
numeric_cols = X.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()
folds = list(StratifiedKFold(n_splits=3, shuffle=True, random_state=42).split(X, y))
p_candidate = np.zeros((len(y), 4), dtype=float)
started = perf_counter()
for fold, (fit_idx, valid_idx) in enumerate(folds, start=1):
    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", SimpleImputer(strategy="median", add_indicator=True), numeric_cols),
            (
                "categorical",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
                        (
                            "onehot",
                            OneHotEncoder(
                                handle_unknown="ignore", sparse_output=False, dtype=np.float32
                            ),
                        ),
                    ]
                ),
                categorical_cols,
            ),
        ],
        sparse_threshold=0,
    )
    x_fit = preprocessor.fit_transform(X.iloc[fit_idx]).astype(np.float32, copy=False)
    x_valid = preprocessor.transform(X.iloc[valid_idx]).astype(np.float32, copy=False)
    model = HistGradientBoostingClassifier(
        learning_rate=0.1,
        max_iter=160,
        max_leaf_nodes=31,
        min_samples_leaf=25,
        l2_regularization=1.0,
        early_stopping=False,
        class_weight="balanced",
        random_state=42,
    )
    with threadpool_limits(limits=2):
        model.fit(x_fit, y[fit_idx])
    p_candidate[valid_idx] = model.predict_proba(x_valid)
    fold_score = f1_score(y[valid_idx], p_candidate[valid_idx].argmax(axis=1), average="macro")
    print(f"Fold {fold}: macro F1={fold_score:.6f}; one-hot features={x_fit.shape[1]}", flush=True)

p_blend = 0.8 * p_base + 0.2 * p_candidate
rows = []
for name, probs in [
    ("Current reference", p_base),
    ("HistGradientBoosting one-hot standalone", p_candidate),
    ("Reference + HistGradientBoosting, fixed 80/20", p_blend),
]:
    pred = probs.argmax(axis=1)
    row = {
        "candidate": name,
        "macro_f1": f1_score(y, pred, average="macro"),
        "weighted_f1": f1_score(y, pred, average="weighted"),
    }
    for fold, (_, valid_idx) in enumerate(folds, start=1):
        row[f"fold_{fold}_macro_f1"] = f1_score(y[valid_idx], pred[valid_idx], average="macro")
        base_fold = f1_score(y[valid_idx], p_base[valid_idx].argmax(axis=1), average="macro")
        row[f"fold_{fold}_delta_vs_reference"] = row[f"fold_{fold}_macro_f1"] - base_fold
    rows.append(row)
results = pd.DataFrame(rows)
display(results.round(6))
results.to_csv(ROOT / "assets/segment-histgb-results.csv", index=False)
for suffix, probs in [("histgb-onehot", p_candidate), ("histgb-onehot-blend", p_blend)]:
    pd.DataFrame(probs, columns=[f"prob_{i}" for i in range(4)]).assign(segment=y).to_csv(
        ROOT / "predictions" / f"segment-{suffix}-oof.csv", index=False
    )
print(f"Elapsed seconds={perf_counter() - started:.1f}. Train OOF only; Test not loaded.")

Fold 1: macro F1=0.738341; one-hot features=166


Fold 2: macro F1=0.736365; one-hot features=166


Fold 3: macro F1=0.742845; one-hot features=166


,candidate,macro_f1,weighted_f1,fold_1_macro_f1,fold_1_delta_vs_reference,fold_2_macro_f1,fold_2_delta_vs_reference,fold_3_macro_f1,fold_3_delta_vs_reference
0,Current reference,0.750895,0.764683,0.749445,0.000000,0.749821,0.000000,0.753390,0.000000
1,HistGradientBoosting one-hot standalone,0.739190,0.751303,0.738341,-0.011105,0.736365,-0.013456,0.742845,-0.010545
2,"Reference + HistGradientBoosting, fixed 80/20",0.751036,0.764605,0.748977,-0.000468,0.750345,0.000524,0.753755,0.000365


Elapsed seconds=225.1. Train OOF only; Test not loaded.
